# 03 · Risk-term counts per filing

**Reads** `stg_cleaned_filings` (dbt) · **Writes** `fact_risk`

Counts how often each risk keyword appears in the Risk Factors section (lemma-based, so *interest rate*,
*Interest Rates* and *supply chain* all match) and stores the section length so dbt can compute a
per-1,000-word **risk density**. Logic: `trading_sentiment/risk_terms.py`.

In [ ]:
%pip install spacy==3.7.2
!python -m spacy download en_core_web_sm

In [ ]:
%restart_python

In [ ]:
# Make the repository's Python package importable (Databricks Git folder: cwd is this notebook's folder)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from trading_sentiment import table

In [ ]:
import spacy
from trading_sentiment.risk_terms import build_matcher, count_risk_terms, summarize

# the parser and NER are not needed for lemmas -> much faster
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
nlp.max_length = 3_000_000
matcher = build_matcher(nlp)

In [ ]:
filings = spark.table(table("stg_cleaned_filings")).where("form_type = '10-K'")
pdf = filings.select("cik", "company_name", "filing_date", "accessionNumber", "form_type",
                     "market_cap", "sector", "risks_text").toPandas()

rows = []
for rec in pdf.itertuples():
    stats = summarize(count_risk_terms(rec.risks_text, nlp, matcher), rec.risks_text)
    rows.append({
        "accession_number": rec.accessionNumber, "cik": rec.cik, "company_name": rec.company_name,
        "filing_date": rec.filing_date, "form_type": rec.form_type,
        **stats, "market_cap": int(rec.market_cap) if rec.market_cap == rec.market_cap else None,
        "sector": rec.sector,
    })
print(len(rows), "filings scored")

`CREATE OR REPLACE` keeps the schema explicit; the write below overwrites the table, so re-running is idempotent.

In [ ]:
from pyspark.sql import functions as F

spark.sql(f"""
CREATE OR REPLACE TABLE {table("fact_risk")} (
    accession_number STRING,
    cik STRING,
    company_name STRING,
    filing_date DATE,
    form_type STRING,
    risk_term_count INT,
    top_terms ARRAY<STRING>,
    risk_word_count INT,
    market_cap LONG,
    sector STRING,
    processed_timestamp TIMESTAMP
) USING DELTA
""")

fact_risk = (spark.createDataFrame(rows, "accession_number string, cik string, company_name string, filing_date date, "
                                          "form_type string, risk_term_count int, top_terms array<string>, "
                                          "risk_word_count int, market_cap long, sector string")
                  .withColumn("processed_timestamp", F.current_timestamp()))
fact_risk.write.format("delta").mode("overwrite").saveAsTable(table("fact_risk"))
display(fact_risk.orderBy(F.desc("risk_term_count")).limit(10))